# Explainability for trained Constrained GCN-LSTM-PPO last-mile models
**Inference / read-only. No training, no checkpoint writes.** Outputs go to `{WORK_DIR}/explain_<preset>/` (new folder).

Run cells top to bottom. **Do NOT run the `pipeline` cell of the training notebook**: it resumes missing seeds/sweeps = training.

Before this notebook: re-run the training notebook's cell 7 (`%%writefile /content/lastmile_rl.py`) if the runtime was recycled. It only rewrites that file.

Known limits (see answer): stored `eval_test.json` tours = best-of-8 sampling + intra-zone 2-opt, not greedy; stored `costs` = raw rollout before 2-opt.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Settings (same values as training notebook)

In [ ]:
WORK_DIR = "/content/drive/MyDrive/lastmile_work"
PRESET = "full"
EXPLAIN_DIR = f"{WORK_DIR}/explain_{PRESET}"   # NEW output folder; runs_*/cache_*/report_* untouched
RUNS = f"{WORK_DIR}/runs_{PRESET}"
CACHE = f"{WORK_DIR}/cache_{PRESET}"
print(EXPLAIN_DIR)

## 3. Check what exists (nothing is assumed)

In [ ]:
from pathlib import Path
import os
print("lastmile_rl.py:", os.path.exists("/content/lastmile_rl.py"), "(if False: re-run training notebook cell 7)")
print("cache:", (Path(CACHE)/"index.json").exists(), "| baselines:", (Path(CACHE)/"baselines.json").exists())
for d in sorted(Path(RUNS).glob("*")):
    if d.is_dir():
        print(f"{d.name:<24}", {f: (d/f).exists() for f in ("train_log.jsonl","best.pt","eval_test.json","eval_test_perturb0.2.json")})

## 4. Write the three scripts to /content

In [ ]:
%%writefile /content/explain_penalties.py
#!/usr/bin/env python3
"""
explain_penalties.py - MACRO-level explanation: how penalty weights (lambda) moved during training.

READ-ONLY on existing results. No training, no evaluation, no model forward pass.
Needs only: json, numpy, matplotlib. (torch only if --mark-best is passed.)

Input : <runs-dir>/<name>_s<seed>/train_log.jsonl   (written by `lastmile_rl.py train`)
Output: <out-dir>/lambda_over_updates.png / .pdf / .csv
        <out-dir>/excess_over_updates.png
        <out-dir>/run_summary.csv

What each train_log.jsonl line holds (verified against cmd_train in lastmile_rl.py):
  update            : 1..U
  lambda            : [n_phases][3] seconds-per-unit-cost for (zone_exit, zone_reentry, far_jump),
                      value AFTER the dual-ascent step of this update (so it is the lambda used by update+1)
  train_excess      : {cost: mean over batch of (episode cost - driver budget), summed over phases}
  val (every eval-every updates only): {ratio_vs_nn, gap_vs_driver, mean_excess{...}, feasible}
Lambda at "update 0" (= --lambda-init) is NOT in the log; --mark-best reads it from best.pt args.

BC logs (bc_*_s*/train_log.jsonl) have no 'lambda' key and are skipped automatically.
Frozen-penalty runs have constant lambda: reported in run_summary.csv, not plotted unless --include-frozen.
"""
import argparse
import csv
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

COSTS = ("zone_exit", "zone_reentry", "far_jump")   # same order as COST_NAMES in lastmile_rl.py


def read_log(path: Path):
    recs = []
    for ln in path.read_text(encoding="utf-8").splitlines():
        if ln.strip():
            r = json.loads(ln)
            if "lambda" in r:
                recs.append(r)
    return recs


def ckpt_info(run_dir: Path):
    """Optional, read-only: step of best.pt and lambda_init. Never writes."""
    try:
        import torch
        ck = torch.load(run_dir / "best.pt", map_location="cpu", weights_only=False)
        a = ck.get("args", {})
        return {"best_step": int(ck["step"]), "lambda_init": a.get("lambda_init"),
                "penalty": a.get("penalty"), "active": a.get("active")}
    except Exception as e:
        print(f"  (no ckpt info for {run_dir.name}: {type(e).__name__})")
        return None


def roll(x, w):
    if w <= 1 or len(x) < w:
        return np.asarray(x, float)
    k = np.ones(w) / w
    return np.convolve(np.pad(x, (w // 2, w - 1 - w // 2), mode="edge"), k, mode="valid")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs-dir", required=True, help="e.g. /content/drive/MyDrive/lastmile_work/runs_full")
    ap.add_argument("--out-dir", required=True, help="NEW folder, e.g. .../explain_full/macro")
    ap.add_argument("--runs", default="", help="comma list of run dir names (default: all with a lambda log)")
    ap.add_argument("--include-frozen", action="store_true", help="also plot constant-lambda runs")
    ap.add_argument("--mark-best", action="store_true", help="read best.pt (read-only, needs torch) to mark selected update")
    ap.add_argument("--smooth", type=int, default=10, help="rolling window for train_excess")
    a = ap.parse_args()

    runs_dir, out = Path(a.runs_dir), Path(a.out_dir)
    want = {s for s in a.runs.split(",") if s}
    logs = sorted(p for p in runs_dir.glob("*/train_log.jsonl") if not want or p.parent.name in want)
    if not logs:
        raise SystemExit(f"no train_log.jsonl under {runs_dir}")
    out.mkdir(parents=True, exist_ok=True)

    runs = {}
    for p in logs:
        recs = read_log(p)
        if not recs:
            print(f"skip {p.parent.name}: no 'lambda' records (behaviour-cloning log?)")
            continue
        lam = np.array([r["lambda"] for r in recs], float)          # [U, P, K]
        upd = np.array([r["update"] for r in recs])
        order = np.argsort(upd, kind="stable")
        runs[p.parent.name] = {
            "dir": p.parent, "upd": upd[order], "lam": lam[order], "recs": [recs[i] for i in order],
            "varies": bool(np.abs(lam - lam[0]).max() > 1e-12),
        }

    # ---- summary table (always, includes frozen runs) ----
    with open(out / "run_summary.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["run", "updates_logged", "phases", "lambda_varies", "cost", "phase",
                    "lambda_first_logged", "lambda_last_logged", "lambda_max"])
        for name, r in runs.items():
            for k, c in enumerate(COSTS):
                for ph in range(r["lam"].shape[1]):
                    s = r["lam"][:, ph, k]
                    w.writerow([name, len(s), r["lam"].shape[1], r["varies"], c, ph + 1,
                                f"{s[0]:.4g}", f"{s[-1]:.4g}", f"{s.max():.4g}"])
    frozen = [n for n, r in runs.items() if not r["varies"]]
    if frozen:
        print("constant lambda (frozen penalty) - not plotted:", ", ".join(frozen))

    plot = {n: r for n, r in runs.items() if r["varies"] or a.include_frozen}
    if not plot:
        raise SystemExit("no run with changing lambda; use --include-frozen to plot constant ones")

    info = {n: (ckpt_info(r["dir"]) if a.mark_best else None) for n, r in plot.items()}

    # ---- tidy CSV ----
    with open(out / "lambda_over_updates.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["run", "update", "phase", "cost", "lambda_after_update"])
        for n, r in plot.items():
            for i, u in enumerate(r["upd"]):
                for ph in range(r["lam"].shape[1]):
                    for k, c in enumerate(COSTS):
                        w.writerow([n, int(u), ph + 1, c, f"{r['lam'][i, ph, k]:.6g}"])

    # ---- figure 1: lambda ----
    R = len(plot)
    fig, axs = plt.subplots(R, 3, figsize=(12, 2.6 * R), squeeze=False, sharex="col")
    for i, (n, r) in enumerate(plot.items()):
        P = r["lam"].shape[1]
        for k, c in enumerate(COSTS):
            ax = axs[i, k]
            for ph in range(P):
                lab = "all stops" if P == 1 else f"phase {ph + 1}/{P}"
                ax.plot(r["upd"], r["lam"][:, ph, k], label=lab, lw=1.4)
            inf = info[n]
            if inf and inf["best_step"] is not None:
                ax.axvline(inf["best_step"], color="k", ls=":", lw=1)
            if i == 0:
                ax.set_title(c)
            if k == 0:
                ax.set_ylabel(f"{n}\nlambda (s / unit cost)")
            if i == R - 1:
                ax.set_xlabel("PPO update")
            ax.grid(alpha=0.3)
            if k == 2:
                ax.legend(fontsize=7, loc="best")
    note = "lambda logged after each dual-ascent step (= value used by next update); update-0 value not logged"
    if a.mark_best:
        note += "; dotted line = update of best.pt"
    fig.suptitle(note, fontsize=8, y=0.995)
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(out / f"lambda_over_updates.{ext}", dpi=150)
    plt.close(fig)

    # ---- figure 2: budget excess (what drives dual ascent) ----
    fig, axs = plt.subplots(R, 1, figsize=(8, 2.6 * R), squeeze=False, sharex=True)
    for i, (n, r) in enumerate(plot.items()):
        ax = axs[i, 0]
        for c in COSTS:
            y = np.array([rec["train_excess"][c] for rec in r["recs"]], float)
            ax.plot(r["upd"], roll(y, a.smooth), label=f"train {c} (roll {a.smooth})", lw=1.2)
            vx = [rec["update"] for rec in r["recs"] if "val" in rec]
            vy = [rec["val"]["mean_excess"][c] for rec in r["recs"] if "val" in rec]
            ax.scatter(vx, vy, s=10, marker="x")
        ax.axhline(0, color="k", lw=0.8)
        ax.set_ylabel(f"{n}\nexcess vs driver budget")
        ax.grid(alpha=0.3)
        if i == 0:
            ax.legend(fontsize=7, ncol=3)
    axs[-1, 0].set_xlabel("PPO update  (x = validation mean_excess, only every eval-every updates)")
    fig.tight_layout()
    fig.savefig(out / "excess_over_updates.png", dpi=150)
    plt.close(fig)
    print(f"wrote -> {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/explain_decision.py
#!/usr/bin/env python3
"""
explain_decision.py - DECISION-level explanation for ONE route of a trained checkpoint.

INFERENCE ONLY. Loads a checkpoint read-only, runs one greedy rollout, recomputes the actor logits of
every step with Policy.evaluate (same function PPO/BC use; no gradients, no optimizer, no saving of weights).
Writes only into --out-dir (a NEW folder). Does not touch runs_*/, cache_*/, report_*/.

Run as a fresh process (!python ...), never %run / import inside the notebook kernel:
lastmile_rl.py must be imported BEFORE torch (OR-Tools load order).

What it prints for the chosen step t:
  * chosen stop + critic value + lambda (from the checkpoint) used to weigh immediate costs
  * a table of the valid candidates (action mask), sorted by actor logit, with
      logit, softmax prob,
      the 6 dynamic features the actor actually receives (dynamic_features() in lastmile_rl.py):
        t_cand/tau          = T[cur,cand] / tau                       (code: row / tau)
        extra_vs_nearest    = max(T[cur,cand] - nearest_unvisited, 0)/tau   (code: np.maximum(row-nearest,0)/tau)
        same_zone           = cand in current zone (0/1)
        reentry             = cand's zone was left earlier with stops remaining (0/1)   (env.open_exited)
        zone_rem            = unvisited fraction of cand's zone                (docstring: zone_remaining_ratio)
        zone_exit_d         = min time from cand's zone to another unvisited zone / tau (docstring: zone_min_exit_dist)
      (names for the first four are MY labels - the code has no string names for them),
      seconds T[cur,cand],
      immediate env costs if this candidate were chosen (exact: env.step on a copy): exit, reentry, far,
      pen_s = sum_k lambda[phase,k]*cost_k  (the penalty seconds PPO subtracts from the reward at this step).
NOT included, because it does not exist in the model: per-feature contributions. The actor is a 2-layer MLP over
[GCN+stats embedding (20 dims, learned), 6 dynamic features, LSTM route context], so logit is NOT a sum of columns.
Treat the feature columns as inputs shown next to the score, not as a causal decomposition.

Example (Colab):
  !python /content/explain_decision.py \
     --cache-dir /content/drive/MyDrive/lastmile_work/cache_full \
     --ckpt /content/drive/MyDrive/lastmile_work/runs_full/frozen_topk_s0/best.pt \
     --split test --route-index 0 --list-steps \
     --out-dir /content/drive/MyDrive/lastmile_work/explain_full/decision
"""
import argparse
import copy
import csv
import os
import sys
from pathlib import Path

sys.path.insert(0, os.environ.get("LASTMILE_DIR", "/content"))   # folder holding lastmile_rl.py
import lastmile_rl as L          # noqa: E402  (must come before torch)
import numpy as np               # noqa: E402
import torch                     # noqa: E402

FEATS = ["t_cand/tau", "extra_vs_nearest", "same_zone", "reentry", "zone_rem", "zone_exit_d"]


def clone_env(env):
    e = copy.copy(env)                       # route / zone_size shared (never mutated by step)
    e.visited, e.zone_left = env.visited.copy(), env.zone_left.copy()
    e.open_exited, e.seq = set(env.open_exited), list(env.seq)
    return e


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cache-dir", required=True)
    ap.add_argument("--ckpt", required=True, help="best.pt or last.pt (opened read-only)")
    ap.add_argument("--split", default="test", choices=("train", "val", "test"))
    ap.add_argument("--route-id", default="", help="route id (as in eval_*.json 'route')")
    ap.add_argument("--route-index", type=int, default=0, help="index into the split if --route-id not given")
    ap.add_argument("--step", type=int, default=-1, help="decision step (0-based). default: first step where greedy != nearest valid stop")
    ap.add_argument("--top", type=int, default=8, help="rows to print (chosen stop always shown)")
    ap.add_argument("--list-steps", action="store_true", help="also print one line per step")
    ap.add_argument("--compare-eval", default="", help="optional eval_*.json: check greedy tour vs stored 'sequence'")
    ap.add_argument("--device", default="cpu", help="cpu = deterministic (as in the script docs)")
    ap.add_argument("--out-dir", required=True)
    a = ap.parse_args()

    dev = torch.device(a.device)
    L.set_seed(0)
    ck = torch.load(a.ckpt, map_location=dev, weights_only=False)            # read-only
    cfg = L.EnvConfig(**ck["env_cfg"])
    policy = L.Policy().to(dev).eval()
    policy.load_state_dict(ck["policy"])
    lam = np.array(ck["lam"], float)                                          # [P, K]
    run_name = f"{ck['args'].get('name', 'run')}_s{ck['args'].get('seed', '?')}"
    print(f"ckpt: {a.ckpt}\n  run={run_name} update={ck['step']} penalty={ck['args'].get('penalty')} env={ck['env_cfg']}")
    print(f"  lambda[phase][zone_exit,zone_reentry,far_jump] = {np.round(lam, 2).tolist()}")

    store = L.RouteStore(a.cache_dir, cfg, dev)
    ids = store.split(a.split)
    rid = a.route_id or ids[a.route_index]
    if rid not in ids:
        raise SystemExit(f"route {rid} not in split '{a.split}'")
    route = store.get(rid)
    with np.load(Path(a.cache_dir) / "routes" / f"{rid}.npz") as z:
        node_ids = [str(s) for s in z["node_ids"]]

    ep = L.run_episode(route, cfg, "greedy", policy)                          # inference only
    with torch.no_grad():
        logits, values = policy.evaluate(route, ep)                           # [S,N], [S]
    logits, values = logits.cpu().numpy(), values.cpu().numpy()
    S, N = logits.shape
    mism = int((logits.argmax(1) != ep.actions).sum())
    lp = torch.log_softmax(torch.tensor(logits), -1).numpy()[np.arange(S), ep.actions]
    print(f"route {rid}: {S} stops, greedy time {ep.total_time/60:.1f} min | check: argmax!=action at {mism}/{S} steps, "
          f"max|logp diff|={np.abs(lp - ep.logp).max():.2e}")
    if mism:
        print("  WARNING: recomputed logits disagree with rollout at some steps (float nondeterminism?). Use --device cpu.")

    # cost / penalty preview per step, using the environment itself (exact, no re-implementation)
    env = L.RouteEnv(route, cfg)
    rows_by_step, summary = [], []
    for t in range(S):
        assert env.cur == int(ep.cur[t]), "replay diverged"
        valid = np.flatnonzero(ep.mask[t])
        ph = env.phase()
        recs = []
        for j in valid:
            tt, c, _ = clone_env(env).step(int(j))
            recs.append((int(j), float(tt), c, float((c * lam[ph]).sum())))
        rows_by_step.append((ph, recs))
        row_t = route.T[env.cur, valid]
        nearest_valid = int(valid[np.argmin(row_t)])
        probs = np.exp(lp[t])
        summary.append({"step": t, "phase": ph + 1, "cur": node_ids[env.cur], "chosen": node_ids[int(ep.actions[t])],
                        "n_valid": int(valid.size), "chosen_is_nearest_valid": int(ep.actions[t]) == nearest_valid,
                        "p_chosen": float(probs), "critic_value": float(values[t])})
        env.step(int(ep.actions[t]))

    if a.step >= 0:
        t = a.step
    else:
        t = next((s["step"] for s in summary if not s["chosen_is_nearest_valid"]), 0)
    if not 0 <= t < S:
        raise SystemExit(f"--step must be in [0,{S-1}]")

    if a.list_steps:
        print("\nstep ph  n_valid  p_chosen  nearest?  cur -> chosen")
        for s in summary:
            print(f"{s['step']:>4} {s['phase']:>2} {s['n_valid']:>8} {s['p_chosen']:>9.3f}  {str(s['chosen_is_nearest_valid']):>8}  "
                  f"{s['cur']} -> {s['chosen']}")

    ph, recs = rows_by_step[t]
    valid = [r[0] for r in recs]
    lg = logits[t][valid]
    pr = np.exp(lg - lg.max()); pr /= pr.sum()
    order = np.argsort(-lg)
    chosen = int(ep.actions[t])
    show = list(order[: a.top])
    cpos = valid.index(chosen)
    if cpos not in show:
        show.append(cpos)
    print(f"\nDECISION step {t} (phase {ph+1}/{cfg.n_phases}): at {node_ids[int(ep.cur[t])]}, "
          f"{len(valid)} valid candidates, critic V={values[t]:.3f}")
    print(f"CHOSEN (greedy = max logit): node {chosen} / stop {node_ids[chosen]}   p={pr[cpos]:.3f}")
    hdr = (f"{'rank':>4} {'node':>5} {'stop_id':>10} {'logit':>8} {'prob':>6} | " + " ".join(f"{f:>16}" for f in FEATS)
           + f" | {'sec':>6} {'exit':>5} {'reent':>5} {'far':>4} {'pen_s':>7}  zone")
    print(hdr)
    out_rows = []
    for i in show:
        j, tt, c, pen = recs[i]
        d = ep.dyn[t, j]
        rank = int(np.where(order == i)[0][0]) + 1
        mark = " <== chosen" if j == chosen else ""
        print(f"{rank:>4} {j:>5} {node_ids[j]:>10} {lg[i]:>8.3f} {pr[i]:>6.3f} | " + " ".join(f"{x:>16.3f}" for x in d)
              + f" | {tt:>6.0f} {c[0]:>5.2f} {c[1]:>5.0f} {c[2]:>4.0f} {pen:>7.1f}  {int(route.zone[j])}{mark}")
        out_rows.append([rank, j, node_ids[j], f"{lg[i]:.5f}", f"{pr[i]:.5f}", *[f"{x:.5f}" for x in d],
                         f"{tt:.1f}", *[f"{x:.4f}" for x in c], f"{pen:.2f}", int(route.zone[j]), j == chosen])
    print("(logit = actor output; masked/invalid stops have logit -1e9 and are not listed. pen_s uses lambda stored in the ckpt.)")

    out = Path(a.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    stem = f"{run_name}_{a.split}_{rid}"
    with open(out / f"{stem}_step{t}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["rank", "node", "stop_id", "logit", "prob", *FEATS, "travel_s", "cost_exit", "cost_reentry",
                    "cost_far", "penalty_s", "zone", "chosen"])
        w.writerows(out_rows)
    with open(out / f"{stem}_steps.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(summary[0]))
        w.writeheader()
        w.writerows(summary)
    print(f"wrote -> {out}/{stem}_step{t}.csv and {stem}_steps.csv")

    if a.compare_eval:
        import json
        rec = next((r for r in json.load(open(a.compare_eval, encoding="utf-8"))["records"] if r["route"] == rid), None)
        if rec is None:
            print(f"--compare-eval: route {rid} not in {a.compare_eval}")
        else:
            raw = list(map(int, ep.sequence))
            opt = L.intra_zone_two_opt(raw, route.zone, route.T)
            st = rec["sequence"]
            div = next((k for k, (x, y) in enumerate(zip(raw, st)) if x != y), None)
            print(f"\nvs stored eval record: greedy raw == stored: {raw == st} | greedy+intra-zone-2opt == stored: {opt == st}"
                  f" | first position where raw greedy differs: {div}")
            if raw != st and opt != st:
                print("  stored tour was NOT produced by plain greedy (pipeline evaluates trained runs with --sample-n 8 + 2-opt);"
                      " this trace explains the greedy policy, not the stored tour.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/explain_route.py
#!/usr/bin/env python3
"""
explain_route.py - ROUTE-level comparison for one route across existing eval_*.json files.

Part A (always, JSON only; needs numpy): reads eval_*.json 'records' and prints, for the selected route,
    time vs driver/nn/nn_2opt/ortools, amazon_score, stored behaviour costs vs driver budget.
Part B (--replay, needs lastmile_rl.py + --cache-dir; NO checkpoint, NO model forward pass):
    replays tours through the environment (run_episode mode="follow") to get the behaviour of the FINAL stored
    sequence, the driver, and the baselines (nn, nn_2opt, ortools from cache baselines.json): totals, per-phase
    costs, zone switches, and a list of cost events (which stop pair caused a zone_exit / re-entry / far_jump).

Why Part B exists: in cmd_evaluate, record["costs"] come from the raw policy rollout (ep.costs) while
record["sequence"] / ["time"] / ["amazon_score"] are computed AFTER best-of-N selection and intra-zone 2-opt.
"costs" can therefore differ from the behaviour of "sequence" (certainly for far_jump; the code docstring
claims exit/re-entry are preserved by the 2-opt). Part B shows the difference instead of assuming it.

Outputs go to --out-dir (NEW folder). Nothing in runs_*/ cache_*/ report_*/ is modified.

Run as a fresh process; with --replay lastmile_rl.py is imported before torch (OR-Tools load order).

Example:
  !python /content/explain_route.py \
     --evals "/content/drive/MyDrive/lastmile_work/runs_full/*/eval_test.json" \
     --cache-dir /content/drive/MyDrive/lastmile_work/cache_full \
     --auto worst_reentry --replay \
     --out-dir /content/drive/MyDrive/lastmile_work/explain_full/route
"""
import argparse
import csv
import glob
import json
import os
import sys
from pathlib import Path

import numpy as np

COSTS = ("zone_exit", "zone_reentry", "far_jump")
BASE = ("driver", "nn", "nn_2opt", "ortools")


def load(p):
    with open(p, encoding="utf-8") as f:
        return json.load(f)


def find(e, rid):
    return next((r for r in e["records"] if r["route"] == rid), None)


def choose_route(first, a):
    recs = first["records"]
    if a.route_id:
        return a.route_id
    key = {
        "worst_reentry": lambda r: r["costs"]["zone_reentry"] - r["driver_costs"]["zone_reentry"],
        "worst_exit": lambda r: r["costs"]["zone_exit"] - r["driver_costs"]["zone_exit"],
        "slowest_vs_driver": lambda r: r["time"] / r["baselines"]["driver"],
    }.get(a.auto)
    if key:
        return max(recs, key=key)["route"]
    return recs[a.route_index]["route"]


def fmt(x, w=8, p=3):
    return f"{x:>{w}.{p}f}" if isinstance(x, (int, float, np.floating)) else f"{str(x):>{w}}"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--evals", nargs="+", required=True, help="glob(s) of eval_*.json (same split/perturb)")
    ap.add_argument("--route-id", default="")
    ap.add_argument("--route-index", type=int, default=0, help="index into records of the first eval file")
    ap.add_argument("--auto", choices=("", "worst_reentry", "worst_exit", "slowest_vs_driver"), default="",
                    help="pick the route from the first eval file instead of by index")
    ap.add_argument("--replay", action="store_true", help="Part B: environment replay (needs --cache-dir)")
    ap.add_argument("--cache-dir", default="")
    ap.add_argument("--out-dir", required=True)
    a = ap.parse_args()

    paths = sorted({p for pat in a.evals for p in glob.glob(pat)})
    if not paths:
        raise SystemExit("no eval files matched")
    evals = [(p, load(p)) for p in paths]
    rid = choose_route(evals[0][1], a)
    out = Path(a.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    print(f"route {rid}   ({len(evals)} eval files)\n")

    # ------------------------------------------------------------------ Part A
    print("PART A - from eval JSON only (costs = raw rollout costs as stored; time/amazon = final sequence)")
    h = (f"{'label':<22}{'seed':>5}{'split':>6}{'pert':>6}{'stops':>6}{'min':>8}"
         f"{'/driver':>9}{'/nn':>8}{'/nn2opt':>9}{'/ortools':>9}{'amazon':>8}")
    print(h)
    rowsA = []
    for p, e in evals:
        r = find(e, rid)
        if r is None:
            print(f"{e['label']:<22} (route not in {p})")
            continue
        b = r["baselines"]
        ratios = [r["time"] / b[k] if k in b else float("nan") for k in BASE[:]]
        print(f"{e['label']:<22}{e['seed']:>5}{e['split']:>6}{e['perturb']:>6}{r['stops']:>6}{r['time']/60:>8.1f}"
              f"{ratios[0]:>9.3f}{ratios[1]:>8.3f}{ratios[2]:>9.3f}{ratios[3]:>9.3f}{r['amazon_score']:>8.4f}")
        row = {"label": e["label"], "seed": e["seed"], "split": e["split"], "perturb": e["perturb"], "mode_field": e["mode"],
               "stops": r["stops"], "time_min": r["time"] / 60, "latency_ms": r["latency_ms"], "amazon_score": r["amazon_score"]}
        row.update({f"time_over_{k}": v for k, v in zip(BASE, ratios)})
        row.update({f"baseline_amazon_{k}": v for k, v in r["baseline_amazon"].items()})
        for c in COSTS:
            row[f"stored_{c}"], row[f"budget_{c}"] = r["costs"][c], r["driver_costs"][c]
            row[f"ok_{c}"] = r["costs"][c] <= r["driver_costs"][c] + 1e-9
        rowsA.append(row)
    print(f"\n{'label':<22}" + "".join(f"{c + ' (got/budget)':>26}" for c in COSTS))
    for row in rowsA:
        print(f"{row['label']:<22}" + "".join(
            f"{row['stored_' + c]:>14.2f}/{row['budget_' + c]:<6.2f}{'ok' if row['ok_' + c] else 'OVER':>4}" for c in COSTS))
    if rowsA:
        with open(out / f"route_{rid}_summary.csv", "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(dict.fromkeys(k for r in rowsA for k in r)))
            w.writeheader()
            w.writerows(rowsA)
    print("\nNOT in eval JSON: per-step data, per-phase costs/budgets (driver_costs is summed over phases), "
          "costs of baseline tours, zone ids / stop ids ('sequence' holds cache node indices), sample_n / zone-2opt flags "
          "('mode' says 'greedy' for any --ckpt run even when best-of-N sampling was used).")

    if not a.replay:
        print(f"\nwrote -> {out}\n(use --replay --cache-dir ... for Part B)")
        return

    # ------------------------------------------------------------------ Part B
    if not a.cache_dir:
        raise SystemExit("--replay needs --cache-dir")
    sys.path.insert(0, os.environ.get("LASTMILE_DIR", "/content"))
    import lastmile_rl as L          # noqa: E402  must precede torch
    import torch                     # noqa: E402

    dev = torch.device("cpu")
    stores = {}

    def store_for(cfg):
        k = tuple(sorted(L.asdict(cfg).items()))
        if k not in stores:
            stores[k] = L.RouteStore(a.cache_dir, cfg, dev)
        return stores[k]

    with np.load(Path(a.cache_dir) / "routes" / f"{rid}.npz") as z:
        node_ids = [str(s) for s in z["node_ids"]]

    print("\nPART B - environment replay of tours (no model). costs recomputed from the tour itself.")
    tours, rows, events, phase_rows = [], [], [], []
    first_cfg = None
    for p, e in evals:
        r = find(e, rid)
        if r is None:
            continue
        cfg = L.EnvConfig(**e["env_cfg"])
        first_cfg = first_cfg or cfg
        st = store_for(cfg)
        route = st.get(rid)
        P = cfg.n_phases
        names = [("driver", route.driver.tolist())] + [(k, st.baselines[rid][k]["seq"]) for k in BASE[1:] if k in st.baselines[rid]]
        names += [(f"{e['label']}[s{e['seed']}] final", r["sequence"])]
        for nm, seq in names:
            if nm in ("driver",) + BASE[1:] and any(t[0] == nm for t in tours):
                continue                      # baselines are identical across eval files
            ep = L.run_episode(route, cfg, "follow", follow=[int(x) for x in seq])
            tot = ep.costs.sum(0)
            zs = route.zone[np.array(seq)]
            switches = int(sum(1 for i in range(1, len(zs)) if zs[i] != zs[i - 1] and zs[i] >= 0 and zs[i - 1] >= 0))
            tours.append((nm, seq))
            rows.append({"tour": nm, "eval_label": e["label"], "seed": e["seed"], "time_min_nominal_T": ep.total_time / 60,
                         **{f"{c}": float(tot[k]) for k, c in enumerate(COSTS)},
                         **{f"budget_{c}": float(route.budget.sum(0)[k]) for k, c in enumerate(COSTS)},
                         "zone_switches": switches, "n_zones_visited": int(len(set(zs[zs >= 0]))),
                         "stored_time_min": r["time"] / 60 if "final" in nm else "",
                         **({f"stored_{c}": r["costs"][c] for c in COSTS} if "final" in nm else {})})
            if P > 1 and ("final" in nm or nm == "driver"):
                cp = ep.costs_by_phase(P)
                for ph in range(P):
                    phase_rows.append({"tour": nm, "phase": ph + 1, **{c: float(cp[ph, k]) for k, c in enumerate(COSTS)},
                                       **{f"budget_{c}": float(route.budget[ph, k]) for k, c in enumerate(COSTS)}})
            for t in range(len(ep.actions)):
                c = ep.costs[t]
                if c.any():
                    events.append({"tour": nm, "step": t, "from": node_ids[int(ep.cur[t])], "to": node_ids[int(ep.actions[t])],
                                   "zone_from": int(route.zone[int(ep.cur[t])]), "zone_to": int(route.zone[int(ep.actions[t])]),
                                   "travel_s": float(ep.travel[t]), "phase": int(ep.phases[t]) + 1,
                                   **{f"cost_{n}": float(c[k]) for k, n in enumerate(COSTS)}})
            if "final" in nm and e["perturb"] == 0 and abs(ep.total_time - r["time"]) > 1e-3:
                print(f"  WARNING {nm}: replay time {ep.total_time:.1f}s != stored {r['time']:.1f}s")

    hdr = f"{'tour':<34}{'min':>8}{'exit':>8}{'reent':>7}{'far':>5}{'switch':>8}{'zones':>6}   stored costs (raw rollout)"
    print(hdr)
    for rw in rows:
        s = ("" if rw["stored_time_min"] == "" else
             f"exit={rw['stored_zone_exit']:.2f} reent={rw['stored_zone_reentry']:.0f} far={rw['stored_far_jump']:.0f}")
        print(f"{rw['tour']:<34}{rw['time_min_nominal_T']:>8.1f}{rw['zone_exit']:>8.2f}{rw['zone_reentry']:>7.0f}"
              f"{rw['far_jump']:>5.0f}{rw['zone_switches']:>8}{rw['n_zones_visited']:>6}   {s}")
    drv = next(rw for rw in rows if rw["tour"] == "driver")
    print(f"driver budget row = driver replay (exit={drv['zone_exit']:.2f}, reentry={drv['zone_reentry']:.0f}, far={drv['far_jump']:.0f}). "
          f"far_jump threshold {first_cfg.farjump_regret_s:.0f}s. 'time' here uses nominal T (perturbed files store perturbed time).")
    if phase_rows:
        print("\nper-phase costs (n_phases>1):")
        for pr in phase_rows:
            print(f"  {pr['tour']:<34} phase {pr['phase']}: " + " ".join(f"{c}={pr[c]:.2f}/{pr['budget_' + c]:.2f}" for c in COSTS))

    for name, data in (("replay_summary", rows), ("replay_phase", phase_rows), ("replay_events", events)):
        if data:
            with open(out / f"route_{rid}_{name}.csv", "w", newline="", encoding="utf-8") as f:
                w = csv.DictWriter(f, fieldnames=list(dict.fromkeys(k for r in data for k in r)))
                w.writeheader()
                w.writerows(data)
    print(f"\nwrote -> {out}  (route_{rid}_summary / replay_summary / replay_phase / replay_events .csv)")


if __name__ == "__main__":
    main()

## 5. Macro: penalty weights (lambda) over training
Reads `train_log.jsonl` only. Frozen runs are flat and listed in `run_summary.csv`.

In [ ]:
!python /content/explain_penalties.py --runs-dir "{RUNS}" --out-dir "{EXPLAIN_DIR}/macro" --mark-best

In [ ]:
from IPython.display import Image, display
for f in ("lambda_over_updates.png", "excess_over_updates.png"):
    p = Path(EXPLAIN_DIR)/"macro"/f
    if p.exists(): display(Image(str(p)))

## 6. Decision: one greedy rollout, candidates + features + logits
Pick a run and route. `STEP = -1` → first step where greedy ≠ nearest valid stop. Greedy trace explains the greedy policy, not necessarily the stored tour (`--compare-eval` checks).

In [ ]:
RUN = "frozen_topk_s0"
SPLIT = "test"
ROUTE_INDEX = 0
STEP = -1
TOP = 8

In [ ]:
!python /content/explain_decision.py --cache-dir "{CACHE}" --ckpt "{RUNS}/{RUN}/best.pt" --split {SPLIT} --route-index {ROUTE_INDEX} --step {STEP} --top {TOP} --list-steps --compare-eval "{RUNS}/{RUN}/eval_{SPLIT}.json" --out-dir "{EXPLAIN_DIR}/decision" 

## 7. Route: compare one route across all eval files + environment replay (no model)

In [ ]:
AUTO = "worst_reentry"   # or "worst_exit", "slowest_vs_driver", "" (then uses ROUTE_ID / index 0)
ROUTE_ID = ""            # e.g. a route id from eval_test.json 'route'

In [ ]:
!python /content/explain_route.py --evals "{RUNS}/*/eval_{SPLIT}.json" --cache-dir "{CACHE}" --auto "{AUTO}" --route-id "{ROUTE_ID}" --replay --out-dir "{EXPLAIN_DIR}/route" 

In [ ]:
import pandas as pd
for f in sorted((Path(EXPLAIN_DIR)/"route").glob("*_replay_summary.csv")):
    display(pd.read_csv(f))

## 8. Optional: greedy, no-2-opt re-evaluation (INFERENCE ONLY)
`evaluate` has no `--out-dir` and writes next to the checkpoint, overwriting `eval_<split>.json`. So the checkpoint is **copied** to a new folder first. Set `RUN_GREEDY_EVAL = True` to run.

In [ ]:
RUN_GREEDY_EVAL = False
if RUN_GREEDY_EVAL:
    dst = f"{EXPLAIN_DIR}/greedy_eval/{RUN}"
    os.makedirs(dst, exist_ok=True)
    import shutil; shutil.copy(f"{RUNS}/{RUN}/best.pt", f"{dst}/best.pt")
    !python /content/lastmile_rl.py evaluate --cache-dir "{CACHE}" --ckpt "{dst}/best.pt" --split {SPLIT} --sample-n 0 --no-zone-2opt --device cpu